# Dual-Stream LLM Generation Example


In [1]:
import json
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoConfig
from datasets import load_dataset
from IPython.display import display, Markdown
import random
from model import DualStreamLLM

# --- Configuration ---
run_dir = 'checkpoints/run_20251027_235405'
checkpoint_path = f'{run_dir}/checkpoint-40000/model.pt'
config_path = f'{run_dir}/training_config.json'
device = 'cuda:1' if torch.cuda.is_available() else 'cpu'
num_examples = 10
max_new_tokens = 50
temperature = 0.7

print(f"Using device: {device}")
print(f"Loading checkpoint: {checkpoint_path}")


Using device: cuda:1
Loading checkpoint: checkpoints/run_20251027_235405/checkpoint-40000/model.pt


In [2]:
device = 'cuda:1' if torch.cuda.is_available() else 'cpu'
device

'cuda:1'

In [ ]:
# Load training configuration
with open(config_path, 'r') as f:
    config = json.load(f)



# Get model parameters from config
base_model_config = AutoConfig.from_pretrained(config['model_name'])
vocab_size = base_model_config.vocab_size
hidden_dim = base_model_config.hidden_size

# Instantiate the model
model = DualStreamLLM(
    model_name=config['model_name'],
    freeze_backbone=config['freeze_backbone'],
    embedding_init_strategy=config['embedding_init_strategy'],
    unembedding_init_strategy=config.get('unembedding_init_strategy'),
    embedding_init_noise_std=config['embedding_init_noise_std'],
    objective=config['objective'],
    shared_embeddings=config['shared_embeddings']
)

print("Model instantiated successfully.")
print(f"  - Objective: {config['objective']}")
print(f"  - Shared Embeddings: {config['shared_embeddings']}")



Model instantiated successfully.
  - Objective: separate
  - Shared Embeddings: False


In [4]:
# Load the checkpoint
checkpoint = torch.load(checkpoint_path, map_location='cpu')

# The model was trained with FSDP, so we need to clean the state dict keys
# by removing the '_fsdp_wrapped_module.' prefix.
model_state_dict = checkpoint['model_state_dict']
cleaned_state_dict = {key.replace('_fsdp_wrapped_module.', ''): value for key, value in model_state_dict.items()}

# Load the cleaned state dict
model.load_state_dict(cleaned_state_dict)

# Set model to evaluation mode and move to device
model.to(device)
model.eval()

print("Model weights loaded successfully.")


Model weights loaded successfully.


In [5]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(config['model_name'])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load dataset
dataset = load_dataset(config['dataset_name'], split='validation', streaming=True)
# Take 2*num_examples to form pairs for prompts A and B
samples = list(dataset.take(num_examples * 2))

print(f"Loaded {len(samples)} samples from the '{config['dataset_name']}' validation set.")


Loaded 20 samples from the 'roneneldan/TinyStories' validation set.


In [6]:
def generate(prompt_A, prompt_B):
    """
    Generates two sequences of text from two prompts using the dual-stream model.
    """
    with torch.no_grad():
        inputs_A = tokenizer(prompt_A, return_tensors='pt', padding=True).to(device)
        inputs_B = tokenizer(prompt_B, return_tensors='pt', padding=True).to(device)

        input_ids_A = inputs_A['input_ids']
        attention_mask_A = inputs_A['attention_mask']
        input_ids_B = inputs_B['input_ids']
        attention_mask_B = inputs_B['attention_mask']

        # Pad inputs to the same length
        max_len = max(input_ids_A.size(1), input_ids_B.size(1))
        if input_ids_A.size(1) < max_len:
            pad_len = max_len - input_ids_A.size(1)
            input_ids_A = torch.cat([input_ids_A, torch.full((1, pad_len), tokenizer.pad_token_id, device=device, dtype=torch.long)], dim=1)
            attention_mask_A = torch.cat([attention_mask_A, torch.zeros((1, pad_len), device=device, dtype=torch.long)], dim=1)
        if input_ids_B.size(1) < max_len:
            pad_len = max_len - input_ids_B.size(1)
            input_ids_B = torch.cat([input_ids_B, torch.full((1, pad_len), tokenizer.pad_token_id, device=device, dtype=torch.long)], dim=1)
            attention_mask_B = torch.cat([attention_mask_B, torch.zeros((1, pad_len), device=device, dtype=torch.long)], dim=1)
        
        # Generation loop
        for _ in range(max_new_tokens):
            logits_A, logits_B = model(input_ids_A, attention_mask_A, input_ids_B, attention_mask_B)

            # Get logits for the last token and apply temperature
            logits_A_scaled = logits_A[:, -1, :] / temperature
            logits_B_scaled = logits_B[:, -1, :] / temperature
            
            # Sample next token
            probs_A = F.softmax(logits_A_scaled, dim=-1)
            probs_B = F.softmax(logits_B_scaled, dim=-1)
            next_token_A = torch.multinomial(probs_A, num_samples=1)
            next_token_B = torch.multinomial(probs_B, num_samples=1)
            
            # Append next token to the input
            input_ids_A = torch.cat([input_ids_A, next_token_A], dim=1)
            input_ids_B = torch.cat([input_ids_B, next_token_B], dim=1)
            attention_mask_A = torch.cat([attention_mask_A, torch.ones((1, 1), device=device, dtype=torch.long)], dim=1)
            attention_mask_B = torch.cat([attention_mask_B, torch.ones((1, 1), device=device, dtype=torch.long)], dim=1)

            # Stop if both streams generate EOS token
            if next_token_A.item() == tokenizer.eos_token_id and next_token_B.item() == tokenizer.eos_token_id:
                break
                
        generated_A = tokenizer.decode(input_ids_A[0], skip_special_tokens=True)
        generated_B = tokenizer.decode(input_ids_B[0], skip_special_tokens=True)
        
        return generated_A, generated_B


In [14]:
# Generate and print examples
for i in range(num_examples):
    prompt_A = ' '.join(samples[i * 2]['text'].split(' ')[:20]) 
    prompt_B = ' '.join(samples[i * 2 + 1]['text'].split(' ')[:20])
    
    generated_A, generated_B = generate(prompt_A, prompt_B)
    
    # Separate prompt from the full generated text
    if generated_A.startswith(prompt_A):
        newly_generated_A = generated_A[len(prompt_A):]
    else:
        newly_generated_A = " (Could not separate prompt from generation)" + generated_A

    if generated_B.startswith(prompt_B):
        newly_generated_B = generated_B[len(prompt_B):]
    else:
        newly_generated_B = " (Could not separate prompt from generation)" + generated_B

    # Display results using Markdown for bolding
    display(Markdown(f"### Example {i+1}/{num_examples}"))
    display(Markdown(f"**Stream A:**"))
    display(Markdown(f"**{prompt_A}**{newly_generated_A}"))
    display(Markdown(f"<br>"))
    display(Markdown(f"**Stream B:**"))
    display(Markdown(f"**{prompt_B}**{newly_generated_B}"))
    display(Markdown(f"---"))


### Example 1/10

**Stream A:**

**Spot. Spot saw the shiny car and said, "Wow, Kitty, your car is so bright and clean!" Kitty smiled and** said, "Yes, Rattle! Let's your friend!" I like to ride your scooter."


"Can I ride the scooter!" said Lily. "Can you turn on, Rattle?"

Spoxy tried to

<br>

**Stream B:**

...

**Once upon a time, there was a big, red barn. Inside the barn, there was a lot of hay. The** barn was very happy. He wanted to take the hay. The people decided to have the animals, so the cow was too small to accept them. The animals were so happy to hear them.

The next day, the cows was so happy

<br>

**Stream B:**

**Once upon a time, there was a big, heavy alligator. He lived near a small pond. He was very hungry** alligator belonged to his friend and they wanted to feed the ducks. They wanted to feed the ducks, but the alligator was too heavy. They were walking the ducks, but the farmer was very hungry. The farmer wanted to help the animal. The farmer

---